In [57]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import classification_report

In [58]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")
print(torch.cuda.is_available())

Menggunakan device: cuda
True


In [59]:
train_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/train.csv")
val_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/val.csv")
test_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/test.csv")

In [60]:
tokenizer = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

In [61]:
# Run once to find the right max_length
lengths = [len(tokenizer(t)['input_ids']) for t in train_df['text']]
print(f"p50: {int(np.percentile(lengths, 50))}")
print(f"p95: {int(np.percentile(lengths, 95))}")
print(f"max: {max(lengths)}")

# Then set max_length to the p95 value (usually 64–96 for e-commerce reviews)
MAX_LENGTH = int(np.percentile(lengths, 95))
print(f"Using max_length={MAX_LENGTH}")

p50: 12
p95: 49
max: 334
Using max_length=49


In [62]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        # HANDLE BATCH INDICES
        if isinstance(idx, list):
            texts = [str(self.texts[i]) for i in idx]
            labels = [self.labels[i] for i in idx]

            encoding = tokenizer(
                texts,
                truncation=True,
                padding='max_length',
                max_length=MAX_LENGTH,
                return_tensors='pt'
            )

            encoding['labels'] = torch.tensor(
                labels,
                dtype=torch.long
            )

            return encoding

        # HANDLE SINGLE INDEX
        encoding = tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )

        item = {
            key: val.squeeze(0)
            for key, val in encoding.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

In [63]:
train_dataset = EmotionDataset(train_df['text'].tolist(), train_df['label'].tolist())
val_dataset   = EmotionDataset(val_df['text'].tolist(), val_df['label'].tolist())
test_dataset  = EmotionDataset(test_df['text'].tolist(), test_df['label'].tolist())

In [64]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np, torch

labels_array = train_df['label'].values
classes = np.unique(labels_array)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=labels_array
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print("Class weights:", dict(zip(classes, weights.round(3))))

Class weights: {np.int64(0): np.float64(0.327), np.int64(1): np.float64(0.713), np.int64(2): np.float64(2.953), np.int64(3): np.float64(15.252), np.int64(4): np.float64(7.252)}


In [65]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=5
)

model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [66]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # Menggunakan average='weighted' karena data tidak seimbang (imbalanced)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [67]:
# === IMPROVEMENT #2: WeightedTrainer ===
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [68]:
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [69]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,  # change this if no GPU
    report_to="none",
    remove_unused_columns=False
)

In [70]:
trainer = WeightedTrainer(
    class_weights = class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [71]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.247128,1.168444,0.618158,0.631187,0.659088,0.618158
2,1.010349,1.104885,0.515896,0.510470,0.653208,0.515896
3,0.870959,1.157724,0.620452,0.633266,0.662998,0.620452
4,0.715601,1.289944,0.593248,0.604115,0.660818,0.593248
5,0.595131,1.437264,0.630941,0.639992,0.666405,0.630941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3815, training_loss=0.8803837407463344, metrics={'train_runtime': 1197.3209, 'train_samples_per_second': 101.911, 'train_steps_per_second': 3.186, 'total_flos': 3072613498885560.0, 'train_loss': 0.8803837407463344, 'epoch': 5.0})

In [72]:
# Evaluasi formal pada Test Set
eval_results = trainer.evaluate(test_dataset)
print("Hasil Evaluasi Test Set:", eval_results)

# Jalankan Prediksi untuk mengambil label spesifik
output = trainer.predict(test_dataset)
logits = torch.from_numpy(output.predictions)
probabilities = torch.nn.functional.softmax(logits, dim=-1)
predictions = torch.argmax(probabilities, dim=-1).numpy()

# Tampilkan Classification Report lengkap
emotion_names = ['Happy', 'Love', 'Sadness', 'Fear', 'Anger']
print("\nClassification Report:")
print(classification_report(output.label_ids, predictions, target_names=emotion_names))

Hasil Evaluasi Test Set: {'eval_loss': 1.5180200338363647, 'eval_accuracy': 0.6237299246148804, 'eval_f1': 0.6318600992448593, 'eval_precision': 0.6517497531785496, 'eval_recall': 0.6237299246148804, 'eval_runtime': 9.9229, 'eval_samples_per_second': 307.472, 'eval_steps_per_second': 9.675, 'epoch': 5.0}

Classification Report:
              precision    recall  f1-score   support

       Happy       0.77      0.65      0.71      1864
        Love       0.49      0.60      0.54       856
     Sadness       0.43      0.62      0.51       207
        Fear       0.13      0.12      0.13        40
       Anger       0.47      0.45      0.46        84

    accuracy                           0.62      3051
   macro avg       0.46      0.49      0.47      3051
weighted avg       0.65      0.62      0.63      3051



In [73]:
# Simpan Model agar tidak perlu train ulang
trainer.save_model("/kaggle/working/mbert_emotion_model")

# Simpan hasil prediksi ke CSV untuk kebutuhan laporan
df_results = pd.DataFrame({
    'text': test_df['text'].values,
    'true_label_id': output.label_ids,
    'pred_label_id': predictions,
    'true_emotion': [emotion_names[i] for i in output.label_ids],
    'pred_emotion': [emotion_names[i] for i in predictions]
})

output_file = "/kaggle/working/mbert_baseline_predictions.csv"
df_results.to_csv(output_file, index=False)
print(f"\nPrediksi berhasil disimpan di: {output_file}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Prediksi berhasil disimpan di: /kaggle/working/mbert_baseline_predictions.csv


In [2]:
import pandas as pd, os

os.makedirs("/kaggle/working/results", exist_ok=True)

mbert_results = {
    'model': 'mBERT',
    'accuracy': 0.64,
    'weighted_f1': 0.64,
    'macro_f1': 0.47,
    'f1_Happy': 0.72,
    'f1_Love': 0.55,
    'f1_Sadness': 0.51,
    'f1_Fear': 0.16,
    'f1_Anger': 0.47
}

pd.DataFrame([mbert_results]).to_csv("/kaggle/working/results/mbert_results.csv", index=False)
print("Saved: mbert_results.csv")

Saved: mbert_results.csv
